# Brazil inflation pressure and monetary policy regimes

Brazil is a useful place to study inflation pressure because the story is not just prices. Exchange-rate shocks, interest-rate cycles, credibility, and inflation persistence all show up in the same monthly data.

This notebook pulls public time series from Banco Central do Brasil, builds a monthly macro panel, estimates inflation-pressure regimes, and checks how the exchange-rate pass-through signal moves over time. The goal is a statistical read of the macro state, not a policy recommendation.


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import warnings
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
PROJECT_DIR = Path.cwd() if Path.cwd().name == "brazil-inflation-monetary-regimes" else Path("notebooks/brazil-inflation-monetary-regimes")
ASSET_DIR = PROJECT_DIR / "assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

BCB_URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{code}/dados"
START_DATE = "01/01/2000"
END_DATE = pd.Timestamp.today().strftime("%d/%m/%Y")

SERIES = {
    "ipca_m": {"code": 433, "label": "IPCA monthly variation"},
    "usd_brl": {"code": 1, "label": "USD/BRL exchange rate"},
    "selic_daily": {"code": 11, "label": "Selic daily rate"},
}

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 180
plt.rcParams["font.family"] = "DejaVu Sans"


## 1. Data from BCB/SGS

The notebook uses three public SGS series: monthly IPCA inflation, daily USD/BRL, and the daily Selic rate. I keep the data source deliberately simple so the project can run without an API key.


In [ ]:
def _fetch_bcb_window(code, name, start, end):
    last_error = None
    for attempt in range(4):
        try:
            response = requests.get(
                BCB_URL.format(code=code),
                params={"formato": "json", "dataInicial": start, "dataFinal": end},
                timeout=30,
            )
            response.raise_for_status()
            if not response.text.strip():
                return pd.DataFrame(columns=["date", name])
            data = response.json()
            break
        except Exception as exc:
            last_error = exc
            if attempt == 3:
                raise
            time.sleep(1.5 * (attempt + 1))
    if not data:
        return pd.DataFrame(columns=["date", name])
    out = pd.DataFrame(data)
    out["date"] = pd.to_datetime(out["data"], dayfirst=True, errors="coerce")
    out[name] = pd.to_numeric(out["valor"].str.replace(",", ".", regex=False), errors="coerce")
    return out.dropna(subset=["date", name])[["date", name]]


def fetch_bcb_series(code, name, start=START_DATE, end=END_DATE, chunked=False):
    if not chunked:
        out = _fetch_bcb_window(code, name, start, end)
    else:
        start_ts = pd.to_datetime(start, dayfirst=True)
        end_ts = pd.to_datetime(end, dayfirst=True)
        frames = []
        cursor = start_ts
        while cursor <= end_ts:
            window_end = min(cursor + pd.DateOffset(years=1) - pd.DateOffset(days=1), end_ts)
            frames.append(_fetch_bcb_window(code, name, cursor.strftime("%d/%m/%Y"), window_end.strftime("%d/%m/%Y")))
            cursor = window_end + pd.DateOffset(days=1)
        out = pd.concat(frames, ignore_index=True)
    if out.empty:
        raise ValueError(f"BCB SGS series {code} returned no rows")
    return out.drop_duplicates("date").sort_values("date").reset_index(drop=True)

ipca = fetch_bcb_series(SERIES["ipca_m"]["code"], "ipca_m")
usd = fetch_bcb_series(SERIES["usd_brl"]["code"], "usd_brl", chunked=True)
selic_daily = fetch_bcb_series(SERIES["selic_daily"]["code"], "selic_daily", chunked=True)

source_summary = pd.DataFrame(
    [
        {"series": "IPCA monthly variation", "code": 433, "start": ipca["date"].min(), "end": ipca["date"].max(), "rows": len(ipca)},
        {"series": "USD/BRL exchange rate", "code": 1, "start": usd["date"].min(), "end": usd["date"].max(), "rows": len(usd)},
        {"series": "Selic daily rate", "code": 11, "start": selic_daily["date"].min(), "end": selic_daily["date"].max(), "rows": len(selic_daily)},
    ]
)
source_summary


## 2. Monthly macro panel

The panel is monthly. IPCA is already monthly. USD/BRL and Selic are converted from daily data into monthly indicators.


In [ ]:
def compound_pct(values):
    values = pd.Series(values).dropna()
    if len(values) == 0:
        return np.nan
    return (np.prod(1 + values / 100) - 1) * 100

ipca_m = ipca.set_index("date").resample("MS").last()

usd_d = usd.set_index("date").sort_index()
usd_daily_returns = np.log(usd_d["usd_brl"]).diff()
usd_d["fx_vol_63d"] = usd_daily_returns.rolling(63).std() * np.sqrt(252) * 100
usd_m = usd_d.resample("MS").agg(usd_brl=("usd_brl", "last"), fx_vol_3m=("fx_vol_63d", "last"))

selic_d = selic_daily.set_index("date").sort_index()
selic_d["selic_annual"] = ((1 + selic_d["selic_daily"] / 100) ** 252 - 1) * 100
selic_m = selic_d.resample("MS").agg(selic_annual=("selic_annual", "last"), selic_daily=("selic_daily", "last"))

panel = ipca_m.join(usd_m, how="inner").join(selic_m, how="inner")
panel["ipca_12m"] = panel["ipca_m"].rolling(12).apply(compound_pct, raw=False)
panel["ipca_3m_ann"] = panel["ipca_m"].rolling(3).apply(lambda x: ((np.prod(1 + x / 100) ** 4) - 1) * 100, raw=False)
panel["inflation_momentum"] = panel["ipca_3m_ann"] - panel["ipca_12m"]
panel["fx_3m_pct"] = np.log(panel["usd_brl"]).diff(3) * 100
panel["fx_12m_pct"] = np.log(panel["usd_brl"]).diff(12) * 100
panel["selic_change_3m"] = panel["selic_annual"].diff(3)
panel["real_policy_rate"] = panel["selic_annual"] - panel["ipca_12m"]
panel["real_rate_gap"] = panel["real_policy_rate"] - panel["real_policy_rate"].rolling(60, min_periods=24).mean()

# Forward 3-month inflation for the pass-through exercise.
next_1 = panel["ipca_m"].shift(-1)
next_2 = panel["ipca_m"].shift(-2)
next_3 = panel["ipca_m"].shift(-3)
panel["ipca_forward_3m"] = ((1 + next_1 / 100) * (1 + next_2 / 100) * (1 + next_3 / 100) - 1) * 100

panel = panel.dropna().reset_index().rename(columns={"date": "month"})
panel.head(), panel.tail()


## 3. Regime model

I fit Gaussian Mixture Models on a training window and select the number of regimes by BIC. The final 20% of months are held out for a simple out-of-sample check.


In [ ]:
MODEL_FEATURES = [
    "ipca_12m",
    "inflation_momentum",
    "fx_3m_pct",
    "fx_vol_3m",
    "selic_annual",
    "selic_change_3m",
    "real_policy_rate",
    "real_rate_gap",
]

panel["split"] = np.where(np.arange(len(panel)) < int(len(panel) * 0.8), "train", "test")
train_mask = panel["split"].eq("train").to_numpy()
test_mask = ~train_mask

scaler = StandardScaler()
X_train = scaler.fit_transform(panel.loc[train_mask, MODEL_FEATURES])
X_test = scaler.transform(panel.loc[test_mask, MODEL_FEATURES])
X_all = scaler.transform(panel[MODEL_FEATURES])

selection_rows = []
models = {}
for k in range(2, 6):
    gmm = GaussianMixture(n_components=k, covariance_type="full", n_init=30, random_state=RANDOM_STATE)
    gmm.fit(X_train)
    models[k] = gmm
    train_labels = gmm.predict(X_train)
    test_labels = gmm.predict(X_test)
    selection_rows.append(
        {
            "k": k,
            "bic_train": gmm.bic(X_train),
            "aic_train": gmm.aic(X_train),
            "train_log_likelihood": gmm.score(X_train),
            "test_log_likelihood": gmm.score(X_test),
            "train_silhouette": silhouette_score(X_train, train_labels),
            "test_silhouette": silhouette_score(X_test, test_labels),
        }
    )

selection = pd.DataFrame(selection_rows)
selected_k = int(selection.loc[selection["bic_train"].idxmin(), "k"])
gmm = models[selected_k]

panel["component"] = gmm.predict(X_all)
panel["assignment_probability"] = gmm.predict_proba(X_all).max(axis=1)

kmeans = KMeans(n_clusters=selected_k, n_init=50, random_state=RANDOM_STATE)
kmeans.fit(X_train)
panel["kmeans_component"] = kmeans.predict(X_all)

inflation_bins = pd.qcut(panel["ipca_12m"], q=selected_k, labels=False, duplicates="drop")
panel["inflation_quantile_regime"] = inflation_bins.astype(int)

selection


## 4. Naming and risk table

The model returns components. I name them after looking at inflation, exchange-rate pressure, Selic, and the real policy rate.


In [ ]:
def regime_stats_for(data, label_col):
    rows = []
    for component, group in data.groupby(label_col):
        rows.append(
            {
                "component": int(component),
                "months": int(len(group)),
                "share": len(group) / len(data),
                "ipca_12m": float(group["ipca_12m"].mean()),
                "inflation_momentum": float(group["inflation_momentum"].mean()),
                "fx_3m_pct": float(group["fx_3m_pct"].mean()),
                "fx_vol_3m": float(group["fx_vol_3m"].mean()),
                "selic_annual": float(group["selic_annual"].mean()),
                "real_policy_rate": float(group["real_policy_rate"].mean()),
                "median_assignment_probability": float(group["assignment_probability"].median()),
            }
        )
    return pd.DataFrame(rows)

regime_stats = regime_stats_for(panel, "component")

def regime_name(row):
    high_infl = row["ipca_12m"] >= regime_stats["ipca_12m"].quantile(0.65)
    high_fx = row["fx_3m_pct"] >= regime_stats["fx_3m_pct"].quantile(0.65)
    high_real = row["real_policy_rate"] >= regime_stats["real_policy_rate"].quantile(0.65)
    low_infl = row["ipca_12m"] <= regime_stats["ipca_12m"].quantile(0.35)
    if high_infl and high_fx:
        return "Inflation/fx pressure"
    if high_infl and high_real:
        return "Tight disinflation"
    if low_infl and not high_real:
        return "Low-pressure easing"
    if high_real:
        return "Restrictive policy"
    return "Sticky middle"

regime_stats["regime"] = regime_stats.apply(regime_name, axis=1)
name_map = dict(zip(regime_stats["component"], regime_stats["regime"]))
panel["regime"] = panel["component"].map(name_map)

ordered = panel.sort_values("month")
ordered["next_regime"] = ordered["regime"].shift(-1)
transition = pd.crosstab(ordered["regime"], ordered["next_regime"], normalize="index").fillna(0)

run_id = (ordered["regime"] != ordered["regime"].shift()).cumsum()
durations = ordered.groupby(run_id).agg(regime=("regime", "first"), duration_months=("regime", "size")).reset_index(drop=True)
duration_stats = durations.groupby("regime").agg(avg_duration_months=("duration_months", "mean"), max_duration_months=("duration_months", "max")).reset_index()
regime_stats = regime_stats.merge(duration_stats, on="regime", how="left").sort_values("ipca_12m", ascending=False).reset_index(drop=True)
regime_stats


## 5. Exchange-rate pass-through signal

This is a small rolling regression, not a structural model. It asks whether recent BRL depreciation helps explain inflation over the next three months after controlling for current inflation and Selic.


In [ ]:
def rolling_pass_through(data, window=60):
    rows = []
    cols = ["fx_3m_pct", "ipca_12m", "selic_annual"]
    usable = data.dropna(subset=["ipca_forward_3m"] + cols).copy()
    for end in range(window, len(usable) + 1):
        sample = usable.iloc[end - window:end]
        y = sample["ipca_forward_3m"].to_numpy()
        X = sample[cols].to_numpy()
        X = np.column_stack([np.ones(len(X)), X])
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        pred = X @ beta
        resid = y - pred
        rmse = float(np.sqrt(np.mean(resid ** 2)))
        rows.append(
            {
                "month": sample["month"].iloc[-1],
                "fx_pass_through_beta": float(beta[1]),
                "inflation_persistence_beta": float(beta[2]),
                "selic_beta": float(beta[3]),
                "rolling_rmse": rmse,
            }
        )
    return pd.DataFrame(rows)

pass_through = rolling_pass_through(panel, window=60)

rng = np.random.default_rng(RANDOM_STATE)
ari_scores = []
base_labels = panel["component"].to_numpy()
train_indices = np.where(train_mask)[0]
for i in range(40):
    sample_idx = rng.choice(train_indices, size=len(train_indices), replace=True)
    boot = GaussianMixture(n_components=selected_k, covariance_type="full", n_init=10, random_state=RANDOM_STATE + i + 1)
    boot.fit(X_all[sample_idx])
    boot_labels = boot.predict(X_all)
    ari_scores.append(adjusted_rand_score(base_labels, boot_labels))

baseline_rows = []
for label_col, label_name in [
    ("component", "GMM selected by BIC"),
    ("kmeans_component", "KMeans same k"),
    ("inflation_quantile_regime", "Inflation quantile baseline"),
]:
    labels = panel[label_col].to_numpy()
    baseline_rows.append(
        {
            "model": label_name,
            "n_regimes": int(pd.Series(labels).nunique()),
            "silhouette_all": float(silhouette_score(X_all, labels)),
            "one_month_persistence": float(np.mean(labels[1:] == labels[:-1])),
        }
    )

baseline_comparison = pd.DataFrame(baseline_rows)
stability_summary = {
    "bootstrap_ari_median": float(np.median(ari_scores)),
    "bootstrap_ari_p10": float(np.percentile(ari_scores, 10)),
    "bootstrap_ari_p90": float(np.percentile(ari_scores, 90)),
}

pass_through.tail(), baseline_comparison, stability_summary


## 6. Report charts


In [ ]:
regime_order = regime_stats.sort_values("ipca_12m", ascending=False)["regime"].tolist()
colors = ["#B91C1C", "#D97706", "#1B6CA8", "#047857", "#6D28D9", "#64748B"]
palette = {name: colors[i % len(colors)] for i, name in enumerate(regime_order)}
plot_df = panel.sort_values("month")

fig, axes = plt.subplots(2, 1, figsize=(13, 8.5), sharex=True)
axes[0].plot(plot_df["month"], plot_df["ipca_12m"], color="#111827", lw=1.4, label="IPCA 12m")
for regime, group in plot_df.groupby("regime"):
    axes[0].scatter(group["month"], group["ipca_12m"], s=22, color=palette[regime], label=regime, alpha=0.82)
axes[0].set_title("Brazil inflation regimes from BCB monthly data")
axes[0].set_ylabel("IPCA 12m (%)")
axes[0].legend(ncol=2, frameon=False, fontsize=9, loc="upper left")
axes[0].axvline(plot_df.loc[test_mask, "month"].min(), color="#475569", ls="--", lw=1)
axes[0].text(plot_df.loc[test_mask, "month"].min(), axes[0].get_ylim()[0], "test window", rotation=90, fontsize=9, color="#475569")

axes[1].plot(plot_df["month"], plot_df["selic_annual"], color="#1B6CA8", lw=1.3, label="Selic annualized")
axes[1].plot(plot_df["month"], plot_df["real_policy_rate"], color="#C46A2B", lw=1.3, label="Ex-post real policy rate")
axes[1].axhline(0, color="#111827", lw=1)
axes[1].set_ylabel("Rate (%)")
axes[1].legend(frameon=False, ncol=2, fontsize=9)
fig.tight_layout()
fig.savefig(ASSET_DIR / "01_macro_regime_timeline.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
axes[0].plot(selection["k"], selection["bic_train"], marker="o", color="#1B6CA8", label="BIC")
axes[0].plot(selection["k"], selection["aic_train"], marker="o", color="#C46A2B", label="AIC")
axes[0].axvline(selected_k, color="#111827", ls="--", lw=1)
axes[0].set_title("Regime count selected on train window")
axes[0].set_xlabel("Number of regimes")
axes[0].set_ylabel("Information criterion")
axes[0].legend(frameon=False)

axes[1].plot(selection["k"], selection["train_log_likelihood"], marker="o", color="#1B6CA8", label="Train")
axes[1].plot(selection["k"], selection["test_log_likelihood"], marker="o", color="#047857", label="Test")
axes[1].axvline(selected_k, color="#111827", ls="--", lw=1)
axes[1].set_title("Out-of-sample log likelihood")
axes[1].set_xlabel("Number of regimes")
axes[1].set_ylabel("Average log likelihood")
axes[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(ASSET_DIR / "02_model_selection.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9.8, 7))
sns.scatterplot(
    data=panel,
    x="real_policy_rate",
    y="inflation_momentum",
    hue="regime",
    size="assignment_probability",
    sizes=(30, 130),
    palette=palette,
    alpha=0.8,
    ax=ax,
)
ax.axhline(0, color="#111827", lw=1)
ax.axvline(0, color="#111827", lw=1)
ax.set_title("Inflation momentum versus real policy rate")
ax.set_xlabel("Ex-post real policy rate (%)")
ax.set_ylabel("3m annualized IPCA minus 12m IPCA (pp)")
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
fig.tight_layout()
fig.savefig(ASSET_DIR / "03_policy_pressure_map.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
transition_plot = transition.reindex(index=regime_order, columns=regime_order).fillna(0)
sns.heatmap(transition_plot, annot=True, fmt=".0%", cmap="Blues", cbar=False, ax=ax)
ax.set_title("One-month transition matrix")
ax.set_xlabel("Next month regime")
ax.set_ylabel("Current regime")
fig.tight_layout()
fig.savefig(ASSET_DIR / "04_transition_matrix.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.8))
ax.plot(pass_through["month"], pass_through["fx_pass_through_beta"], color="#B91C1C", lw=1.6)
ax.axhline(0, color="#111827", lw=1)
ax.set_title("Rolling exchange-rate pass-through signal")
ax.set_xlabel("")
ax.set_ylabel("Coefficient on 3m USD/BRL change")
ax.text(pass_through["month"].min(), ax.get_ylim()[1] * 0.86, "60-month rolling regression", fontsize=10, color="#475569")
fig.tight_layout()
fig.savefig(ASSET_DIR / "05_rolling_pass_through.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))
table_df = regime_stats.copy()
table_df["IPCA 12m"] = table_df["ipca_12m"].map(lambda x: f"{x:.1f}%")
table_df["FX 3m"] = table_df["fx_3m_pct"].map(lambda x: f"{x:.1f}%")
table_df["Selic"] = table_df["selic_annual"].map(lambda x: f"{x:.1f}%")
table_df["Real rate"] = table_df["real_policy_rate"].map(lambda x: f"{x:.1f}%")
table_df["Median p"] = table_df["median_assignment_probability"].map(lambda x: f"{x:.0%}")
table_df["Avg duration"] = table_df["avg_duration_months"].map(lambda x: f"{x:.1f}m")
cols = ["regime", "months", "IPCA 12m", "FX 3m", "Selic", "Real rate", "Avg duration", "Median p"]
ax.axis("off")
table = ax.table(cellText=table_df[cols].values, colLabels=cols, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.4)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#CBD5E1")
    if row == 0:
        cell.set_facecolor("#E2E8F0")
        cell.set_text_props(weight="bold")
ax.set_title("Regime profile table", pad=18)
fig.tight_layout()
fig.savefig(ASSET_DIR / "06_regime_profile_table.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
base = baseline_comparison.sort_values("silhouette_all", ascending=True)
axes[0].barh(base["model"], base["silhouette_all"], color=["#1B6CA8" if "GMM" in m else "#94A3B8" for m in base["model"]])
axes[0].set_title("Baseline comparison")
axes[0].set_xlabel("Silhouette score")
for i, row in enumerate(base.itertuples()):
    axes[0].text(row.silhouette_all + 0.005, i, f"persistence {row.one_month_persistence:.0%}", va="center", fontsize=9)

sns.histplot(ari_scores, bins=16, color="#047857", ax=axes[1])
axes[1].axvline(np.median(ari_scores), color="#111827", ls="--", lw=1)
axes[1].set_title("Bootstrap stability")
axes[1].set_xlabel("Adjusted Rand index")
axes[1].set_ylabel("Bootstrap refits")
fig.tight_layout()
fig.savefig(ASSET_DIR / "07_baseline_stability.png", bbox_inches="tight")
plt.show()


## 7. Readout

The final cell writes the headline numbers used by the README.


In [ ]:
latest = panel.sort_values("month").iloc[-1]
selected_row = selection[selection["k"].eq(selected_k)].iloc[0]
gmm_baseline = baseline_comparison[baseline_comparison["model"].eq("GMM selected by BIC")].iloc[0]
latest_pass = pass_through.dropna().iloc[-1]

summary = {
    "data_start": panel["month"].min().strftime("%Y-%m-%d"),
    "data_end": panel["month"].max().strftime("%Y-%m-%d"),
    "analysis_months": int(len(panel)),
    "train_months": int(train_mask.sum()),
    "test_months": int(test_mask.sum()),
    "selected_regimes": selected_k,
    "latest_regime": str(latest["regime"]),
    "latest_regime_probability": float(latest["assignment_probability"]),
    "latest_ipca_12m": float(latest["ipca_12m"]),
    "latest_selic_annual": float(latest["selic_annual"]),
    "latest_real_policy_rate": float(latest["real_policy_rate"]),
    "latest_fx_3m_pct": float(latest["fx_3m_pct"]),
    "latest_pass_through_beta": float(latest_pass["fx_pass_through_beta"]),
    "median_assignment_probability": float(panel["assignment_probability"].median()),
    "test_median_assignment_probability": float(panel.loc[test_mask, "assignment_probability"].median()),
    "gmm_silhouette": float(gmm_baseline["silhouette_all"]),
    "gmm_one_month_persistence": float(gmm_baseline["one_month_persistence"]),
    "bootstrap_ari_median": stability_summary["bootstrap_ari_median"],
    "bootstrap_ari_p10": stability_summary["bootstrap_ari_p10"],
    "bootstrap_ari_p90": stability_summary["bootstrap_ari_p90"],
    "train_bic_selected": float(selected_row["bic_train"]),
    "test_log_likelihood_selected": float(selected_row["test_log_likelihood"]),
}

(PROJECT_DIR / "results_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary
